# Study the laplacian regression on the Larged Uncrompressed Graph

In [3]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import networkx as nx

from scipy import sparse
from scipy.sparse.linalg import spsolve

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 1. FILE LOCATIONS
# ============================================================

BASE_DIR = Path.cwd()

FULL_GRAPH_FILE = (
    BASE_DIR
    / "largest_river_component.pkl"
)

MASTER_GRAPH_FILE = (
    BASE_DIR
    / "master_compressed_river_graph.pkl"
)

HOURLY_FILE = (
    BASE_DIR
    / "usgs_all_stations_hourly.csv"
)


print("Current directory:")
print(BASE_DIR)


# ============================================================
# 2. CHECK FILES EXIST
# ============================================================

required_files = [
    FULL_GRAPH_FILE,
    MASTER_GRAPH_FILE,
    HOURLY_FILE
]


for file in required_files:

    print(
        file.name,
        "->",
        "FOUND"
        if file.exists()
        else "MISSING"
    )


missing_files = [
    file
    for file in required_files
    if not file.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Missing files:\n"
        +
        "\n".join(
            str(file)
            for file in missing_files
        )
    )


# ============================================================
# 3. LOAD UNCOMPRESSED LARGEST COMPONENT
# ============================================================

with open(
    FULL_GRAPH_FILE,
    "rb"
) as f:

    G_full = pickle.load(f)


print("\n" + "=" * 60)
print("FULL / UNCOMPRESSED GRAPH")
print("=" * 60)

print(
    "Directed:",
    G_full.is_directed()
)

print(
    "Nodes:",
    G_full.number_of_nodes()
)

print(
    "Edges:",
    G_full.number_of_edges()
)


if G_full.is_directed():

    print(
        "Weak components:",
        nx.number_weakly_connected_components(
            G_full
        )
    )

else:

    print(
        "Connected components:",
        nx.number_connected_components(
            G_full
        )
    )


# ============================================================
# 4. LOAD MASTER COMPRESSED GRAPH
#
# We are NOT using this for the new Laplacian.
# We only need its station/support metadata.
# ============================================================

with open(
    MASTER_GRAPH_FILE,
    "rb"
) as f:

    G_master = pickle.load(f)


print("\n" + "=" * 60)
print("MASTER METADATA GRAPH")
print("=" * 60)

print(
    "Nodes:",
    G_master.number_of_nodes()
)

print(
    "Edges:",
    G_master.number_of_edges()
)


# ============================================================
# 5. LOAD HOURLY USGS DATA
# ============================================================

hourly = pd.read_csv(
    HOURLY_FILE
)


hourly["time"] = pd.to_datetime(
    hourly["time"],
    utc=True,
    errors="coerce"
)


print("\n" + "=" * 60)
print("HOURLY DATA")
print("=" * 60)

print(
    "Rows:",
    len(hourly)
)

print(
    "Stations:",
    hourly["station_id"].nunique()
)

print(
    "Columns:"
)

print(
    hourly.columns.tolist()
)


# ============================================================
# 6. BUILD STATION -> GRAPH NODE MAPPING
# ============================================================

station_to_node = {}


for node, data in G_master.nodes(
    data=True
):

    clusters = data.get(
        "usgs_clusters",
        []
    )


    for cluster in clusters:

        stations = cluster.get(
            "member_stations",
            []
        )


        # fallback
        if not stations:

            anchor = cluster.get(
                "anchor_station"
            )

            if anchor is not None:

                stations = [
                    anchor
                ]


        for station in stations:

            station_to_node[
                station
            ] = node


print(
    "\nWQ stations mapped:",
    len(station_to_node)
)


# ============================================================
# 7. IDENTIFY WQ NODES
# ============================================================

all_wq_nodes = {

    node

    for node, data
    in G_master.nodes(
        data=True
    )

    if (
        data.get(
            "usgs_station_count",
            0
        ) > 0

        or

        len(
            data.get(
                "usgs_clusters",
                []
            )
        ) > 0
    )
}


# ============================================================
# 8. IDENTIFY SUPPORT-GAUGE NODES
# ============================================================

support_nodes = {

    node

    for node, data
    in G_master.nodes(
        data=True
    )

    if (
        data.get(
            "support_gauge_count",
            0
        ) > 0

        or

        len(
            data.get(
                "support_gauges",
                []
            )
        ) > 0
    )
}


print(
    "WQ graph nodes:",
    len(all_wq_nodes)
)

print(
    "Support-gauge nodes:",
    len(support_nodes)
)


# ============================================================
# 9. VERIFY THEY EXIST IN FULL GRAPH
# ============================================================

full_nodes = set(
    G_full.nodes()
)


missing_wq = (
    all_wq_nodes
    - full_nodes
)

missing_support = (
    support_nodes
    - full_nodes
)


print("\n" + "=" * 60)
print("FULL-GRAPH MONITORING CHECK")
print("=" * 60)


print(
    "WQ nodes present:",
    len(
        all_wq_nodes
        &
        full_nodes
    ),
    "/",
    len(all_wq_nodes)
)


print(
    "Support nodes present:",
    len(
        support_nodes
        &
        full_nodes
    ),
    "/",
    len(support_nodes)
)


print(
    "\nMissing WQ nodes:",
    sorted(
        missing_wq
    )
)

print(
    "Missing support nodes:",
    sorted(
        missing_support
    )
)


# ============================================================
# 10. ATTACH GRAPH NODE TO HOURLY DATA
# ============================================================

hourly["node"] = (
    hourly["station_id"]
    .map(
        station_to_node
    )
)


mapped_hourly = (
    hourly[
        hourly["node"]
        .notna()
    ]
    .copy()
)


mapped_hourly[
    "node"
] = (
    mapped_hourly[
        "node"
    ]
    .astype(int)
)


print("\n" + "=" * 60)
print("READY FOR MODELING")
print("=" * 60)

print(
    "Mapped hourly rows:",
    len(mapped_hourly)
)

print(
    "Mapped WQ graph nodes:",
    mapped_hourly[
        "node"
    ].nunique()
)

print(
    "Full graph nodes available to Laplacian:",
    G_full.number_of_nodes()
)

Current directory:
/Users/andrewchincea/TDA_Lab_Practice/rivernets/making_the_graph
largest_river_component.pkl -> FOUND
master_compressed_river_graph.pkl -> FOUND
usgs_all_stations_hourly.csv -> FOUND

FULL / UNCOMPRESSED GRAPH
Directed: True
Nodes: 67401
Edges: 67400
Weak components: 1

MASTER METADATA GRAPH
Nodes: 67
Edges: 66

HOURLY DATA
Rows: 234095
Stations: 28
Columns:
['station_id', 'time', 'DO concentration', 'Water temperature', 'Specific conductance', 'Turbidity', 'pH', 'Discharge', 'Stage']

WQ stations mapped: 28
WQ graph nodes: 26
Support-gauge nodes: 3

FULL-GRAPH MONITORING CHECK
WQ nodes present: 26 / 26
Support nodes present: 3 / 3

Missing WQ nodes: []
Missing support nodes: []

READY FOR MODELING
Mapped hourly rows: 234095
Mapped WQ graph nodes: 26
Full graph nodes available to Laplacian: 67401
